# Experiment 2: Classifier Variation

Tests different classifiers on the same wav2vec2-BASE L12 features.

Classifiers: MLP_3layer, SVM, Random Forest, XGBoost, LSTM, Transformer.

Tip: set `TARGET_DURATION` to the best duration found in Exp 1. If Exp 1 is not done yet, use 6 (the original baseline).

In [ ]:
# ============================================================
# CONFIG
# ============================================================
TARGET_DURATION = 6     # seconds -- match best result from Exp 1

# Toggle which classifiers to run
RUN_MLP   = True
RUN_SVM   = True
RUN_RF    = True
RUN_XGB   = True
RUN_LSTM  = True
RUN_ATTN  = True

# Feature extractor config (fixed)
MODEL_SIZE  = 'base'
LAYER_INDEX = 11

# Shared training hyperparams
EPOCHS        = 50
BATCH_SIZE    = 32
LEARNING_RATE = 0.001
DROPOUT       = 0.3

print('Config loaded.')

In [ ]:
import os, time, json, math, random
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import to_MLP_Utility as utl
import importlib; importlib.reload(utl)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')
    XGB_AVAILABLE = False
    RUN_XGB = False

CWD          = os.getcwd()
DATASETS_DIR = os.path.join(CWD, 'datasets_separated')
EXP1_PT_DIR  = os.path.join(CWD, 'exp1_results', f'{TARGET_DURATION}s')
OUTPUT_DIR   = os.path.join(CWD, 'exp2_results', f'{TARGET_DURATION}s')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Step 1: Load Features

Tries to load pooled features saved by Exp 1 first.
If not found, re-runs the full preprocessing pipeline.

In [ ]:
pooled_train_path = os.path.join(EXP1_PT_DIR, 'pooled_train.pt')
pooled_val_path   = os.path.join(EXP1_PT_DIR, 'pooled_val.pt')
pooled_test_path  = os.path.join(EXP1_PT_DIR, 'pooled_test.pt')

if os.path.exists(pooled_train_path):
    print(f'Loading pre-computed features from Exp 1 ({TARGET_DURATION}s)...')
    tr = torch.load(pooled_train_path)
    vl = torch.load(pooled_val_path)
    te = torch.load(pooled_test_path)
    train_X, train_y = tr['features'], tr['labels']
    val_X,   val_y   = vl['features'], vl['labels']
    test_X,  test_y  = te['features'], te['labels']
    print('Loaded from Exp 1 cache.')
else:
    print('Exp 1 cache not found. Re-running preprocessing...')
    print('Tip: run exp1_duration_variation.ipynb first to save time.')

    # --- re-run preprocessing (same as Exp 1) ---
    TRESHOLD_LOW = TARGET_DURATION / 3

    def load_all_paths(split, label, datasets_dir):
        name = f'{label}_{split}'
        folder = os.path.join(datasets_dir, name)
        outliers = pd.read_csv(os.path.join(folder, 'outliers.csv'))
        below    = pd.read_csv(os.path.join(folder, f'{name}_below_treshold.csv'))
        for df in [outliers, below]:
            if 'Unnamed: 0' in df.columns:
                df.drop('Unnamed: 0', axis=1, inplace=True)
        combined = pd.concat([outliers, below], ignore_index=True)
        combined = combined.dropna(subset=['file path', 'duration'])
        combined['duration'] = combined['duration'].astype(float)
        return combined

    def re_separate(df, target_duration, low):
        need_slice = df[df['duration'] >= target_duration].reset_index(drop=True)
        need_pad   = df[(df['duration'] >= low) & (df['duration'] < target_duration)].reset_index(drop=True)
        return need_slice, need_pad

    def slice_and_pad(need_slice_df, need_pad_df, target_duration, low, sample_rate=16000):
        max_s = int(sample_rate * target_duration)
        min_s = int(sample_rate * low)
        chunks = []
        for _, row in need_slice_df.iterrows():
            try:
                waveform, sr = torchaudio.load(row['file path'])
                if sr != sample_rate:
                    waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
                n = math.ceil(waveform.size(1) / max_s)
                for i in range(n):
                    chunk = waveform[:, i*max_s:(i+1)*max_s]
                    if chunk.size(1) < min_s: continue
                    if chunk.size(1) < max_s:
                        chunk = torch.nn.functional.pad(chunk, (0, max_s - chunk.size(1)))
                    chunks.append(chunk)
            except Exception as e:
                print(f'  SKIP {row["file path"]}: {e}')
        for _, row in need_pad_df.iterrows():
            try:
                waveform, sr = torchaudio.load(row['file path'])
                if sr != sample_rate:
                    waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
                if waveform.size(1) > max_s: waveform = waveform[:, :max_s]
                chunk = torch.nn.functional.pad(waveform, (0, max_s - waveform.size(1)))
                chunks.append(chunk)
            except Exception as e:
                print(f'  SKIP {row["file path"]}: {e}')
        return chunks

    def build_labeled(real_chunks, fake_chunks, shuffle=True):
        dataset = [(w, 0) for w in real_chunks] + [(w, 1) for w in fake_chunks]
        if shuffle: random.seed(42); random.shuffle(dataset)
        return dataset

    def pool_feature(tensor):
        t = tensor if isinstance(tensor, torch.Tensor) else tensor[0]
        if t.ndim == 3: t = t.squeeze(0)
        return t.mean(dim=0).numpy()

    use_gpu = torch.cuda.is_available()
    extract_fn = utl.feature_extractor_gpu if use_gpu else utl.feature_extractor

    splits = ['training', 'validation', 'testing']
    labels_list = ['real', 'fake']
    all_dfs = {f'{l}_{s}': load_all_paths(s, l, DATASETS_DIR) for s in splits for l in labels_list}

    def process_split(split_name, val_split_name):
        s_r, p_r = re_separate(all_dfs[f'real_{split_name}'], TARGET_DURATION, TRESHOLD_LOW)
        s_f, p_f = re_separate(all_dfs[f'fake_{split_name}'], TARGET_DURATION, TRESHOLD_LOW)
        real_chunks = slice_and_pad(s_r, p_r, TARGET_DURATION, TRESHOLD_LOW)
        fake_chunks = slice_and_pad(s_f, p_f, TARGET_DURATION, TRESHOLD_LOW)
        seq = build_labeled(real_chunks, fake_chunks, shuffle=(split_name == 'training'))
        waveforms, lbls = zip(*seq)
        print(f'  Extracting {val_split_name} features...')
        feats_raw = extract_fn(list(waveforms), model_size=MODEL_SIZE, layer_index=LAYER_INDEX)
        X = np.array([pool_feature(f) for f in feats_raw])
        y = np.array(lbls)
        return X, y

    train_X, train_y = process_split('training',   'train')
    val_X,   val_y   = process_split('validation', 'val')
    test_X,  test_y  = process_split('testing',    'test')

print(f'train: {train_X.shape}, val: {val_X.shape}, test: {test_X.shape}')

In [ ]:
# SMOTE on training set
sm = SMOTE(random_state=42)
train_X_bal, train_y_bal = sm.fit_resample(train_X, train_y)
print(f'After SMOTE: {train_X_bal.shape}')

# Scaler for SVM (SVM is sensitive to scale)
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X_bal)
val_X_scaled   = scaler.transform(val_X)
test_X_scaled  = scaler.transform(test_X)

# Results accumulator
all_results = {}

## Classifier 1: MLP_3layer [384, 192, 96]

In [ ]:
if RUN_MLP:
    print('=== MLP_3layer ===')

    class ResearchMLP(nn.Module):
        def __init__(self, input_size, hidden_sizes, output_size, dropout):
            super().__init__()
            layers = []
            prev = input_size
            for i, h in enumerate(hidden_sizes):
                layers += [nn.Linear(prev, h), nn.ReLU()]
                if i < len(hidden_sizes) - 1:
                    layers.append(nn.Dropout(dropout))
                prev = h
            layers.append(nn.Linear(prev, output_size))
            self.network = nn.Sequential(*layers)
        def forward(self, x): return self.network(x)

    HIDDEN = [384, 192, 96]
    model  = ResearchMLP(768, HIDDEN, 1, DROPOUT)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    ds_train = TensorDataset(torch.FloatTensor(train_X_bal), torch.FloatTensor(train_y_bal))
    ds_val   = TensorDataset(torch.FloatTensor(val_X),       torch.FloatTensor(val_y))
    ds_test  = TensorDataset(torch.FloatTensor(test_X),      torch.FloatTensor(test_y))

    best_val_loss = float('inf'); patience_counter = 0; patience = 5
    start = time.perf_counter()

    for epoch in range(EPOCHS):
        model.train()
        for Xb, yb in DataLoader(ds_train, BATCH_SIZE, shuffle=True):
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb.unsqueeze(1))
            loss.backward(); optimizer.step()

        model.eval()
        val_losses, val_accs = [], []
        with torch.no_grad():
            for Xb, yb in DataLoader(ds_val, BATCH_SIZE):
                out  = model(Xb)
                val_losses.append(criterion(out, yb.unsqueeze(1)).item())
                val_accs.append(((torch.sigmoid(out) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())
        avg_vl = np.mean(val_losses)
        print(f'  Epoch {epoch+1} | Val Loss: {avg_vl:.4f} | Val Acc: {np.mean(val_accs):.4f}')
        if avg_vl < best_val_loss:
            best_val_loss = avg_vl; patience_counter = 0
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'mlp_best.pth'))
        else:
            patience_counter += 1
            if patience_counter >= patience: print('  Early stop.'); break

    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'mlp_best.pth')))
    model.eval()
    test_accs = []
    with torch.no_grad():
        for Xb, yb in DataLoader(ds_test, BATCH_SIZE):
            test_accs.append(((torch.sigmoid(model(Xb)) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())

    runtime = time.perf_counter() - start
    all_results['MLP_3layer'] = {
        'val_accuracy': np.mean(val_accs), 'test_accuracy': np.mean(test_accs), 'runtime_seconds': runtime
    }
    print(f'  Test acc: {np.mean(test_accs):.4f} | Time: {runtime:.1f}s')

## Classifier 2: SVM

In [ ]:
if RUN_SVM:
    print('=== SVM (RBF kernel) ===')
    start = time.perf_counter()

    svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm.fit(train_X_scaled, train_y_bal)

    val_acc  = accuracy_score(val_y,  svm.predict(val_X_scaled))
    test_acc = accuracy_score(test_y, svm.predict(test_X_scaled))
    runtime  = time.perf_counter() - start

    all_results['SVM'] = {'val_accuracy': val_acc, 'test_accuracy': test_acc, 'runtime_seconds': runtime}
    print(f'  Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f} | Time: {runtime:.1f}s')

    # Save model for Exp 3
    import joblib
    joblib.dump(svm,    os.path.join(OUTPUT_DIR, 'svm_model.pkl'))
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))

## Classifier 3: Random Forest

In [ ]:
if RUN_RF:
    print('=== Random Forest ===')
    start = time.perf_counter()

    rf = RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=-1, random_state=42)
    rf.fit(train_X_bal, train_y_bal)

    val_acc  = accuracy_score(val_y,  rf.predict(val_X))
    test_acc = accuracy_score(test_y, rf.predict(test_X))
    runtime  = time.perf_counter() - start

    all_results['RandomForest'] = {'val_accuracy': val_acc, 'test_accuracy': test_acc, 'runtime_seconds': runtime}
    print(f'  Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f} | Time: {runtime:.1f}s')

    import joblib
    joblib.dump(rf, os.path.join(OUTPUT_DIR, 'rf_model.pkl'))

## Classifier 4: XGBoost

In [ ]:
if RUN_XGB and XGB_AVAILABLE:
    print('=== XGBoost ===')
    start = time.perf_counter()

    xgb_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        n_jobs=-1, random_state=42
    )
    xgb_model.fit(
        train_X_bal, train_y_bal,
        eval_set=[(val_X, val_y)],
        verbose=50
    )

    val_acc  = accuracy_score(val_y,  xgb_model.predict(val_X))
    test_acc = accuracy_score(test_y, xgb_model.predict(test_X))
    runtime  = time.perf_counter() - start

    all_results['XGBoost'] = {'val_accuracy': val_acc, 'test_accuracy': test_acc, 'runtime_seconds': runtime}
    print(f'  Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f} | Time: {runtime:.1f}s')

    xgb_model.save_model(os.path.join(OUTPUT_DIR, 'xgb_model.json'))

## Classifier 5: LSTM

Needs frame-level features [T, 768] -- NOT mean-pooled.
Re-extracts features without pooling.

In [ ]:
if RUN_LSTM:
    print('=== Re-extracting frame-level features for LSTM ===')
    use_gpu = torch.cuda.is_available()
    extract_fn = utl.feature_extractor_gpu if use_gpu else utl.feature_extractor

    # Try to load from cache first
    lstm_cache_train = os.path.join(OUTPUT_DIR, 'lstm_features_train.pt')
    lstm_cache_val   = os.path.join(OUTPUT_DIR, 'lstm_features_val.pt')
    lstm_cache_test  = os.path.join(OUTPUT_DIR, 'lstm_features_test.pt')

    if os.path.exists(lstm_cache_train):
        print('Loading LSTM features from cache...')
        lstm_train = torch.load(lstm_cache_train)
        lstm_val   = torch.load(lstm_cache_val)
        lstm_test  = torch.load(lstm_cache_test)
        lstm_train_feats, lstm_train_labels = lstm_train['features'], lstm_train['labels']
        lstm_val_feats,   lstm_val_labels   = lstm_val['features'],   lstm_val['labels']
        lstm_test_feats,  lstm_test_labels  = lstm_test['features'],  lstm_test['labels']
    else:
        print('Cache not found. Need audio waveforms for re-extraction.')
        print('Please run Exp 1 step 1-4 first to get waveform lists.')
        print('This cell assumes train_seq, val_seq, test_seq are in memory from Exp 1.')
        print('Skipping LSTM if waveforms not available.')
        RUN_LSTM = False

In [ ]:
if RUN_LSTM:
    # frame-level features: shape [T, 768] per sample
    # Pad/truncate to fixed T for batching
    def pad_to_fixed_T(tensor_list, T_max=None):
        seqs = []
        for t in tensor_list:
            if isinstance(t, list): t = t[0]
            if t.ndim == 3: t = t.squeeze(0)  # [T, 768]
            seqs.append(t)
        if T_max is None:
            T_max = max(s.shape[0] for s in seqs)
        padded = []
        for s in seqs:
            T_cur = s.shape[0]
            if T_cur >= T_max:
                padded.append(s[:T_max])
            else:
                pad = torch.zeros(T_max - T_cur, s.shape[1])
                padded.append(torch.cat([s, pad], dim=0))
        return torch.stack(padded), T_max

    print('Padding sequences to fixed length...')
    X_lstm_train, T_MAX = pad_to_fixed_T(lstm_train_feats)
    X_lstm_val,   _     = pad_to_fixed_T(lstm_val_feats,  T_MAX)
    X_lstm_test,  _     = pad_to_fixed_T(lstm_test_feats, T_MAX)
    y_lstm_train = torch.FloatTensor(lstm_train_labels)
    y_lstm_val   = torch.FloatTensor(lstm_val_labels)
    y_lstm_test  = torch.FloatTensor(lstm_test_labels)
    print(f'LSTM input shape: {X_lstm_train.shape}')  # [N, T, 768]

In [ ]:
if RUN_LSTM:
    class LSTMClassifier(nn.Module):
        def __init__(self, input_size=768, hidden_size=256, num_layers=2, dropout=0.3):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size, hidden_size, num_layers,
                batch_first=True, dropout=dropout
            )
            self.classifier = nn.Linear(hidden_size, 1)

        def forward(self, x):  # x: [batch, T, 768]
            _, (hn, _) = self.lstm(x)  # hn: [num_layers, batch, hidden]
            return self.classifier(hn[-1])  # last layer hidden state

    lstm_model = LSTMClassifier()
    lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
    criterion      = nn.BCEWithLogitsLoss()

    ds_train_lstm = TensorDataset(X_lstm_train, y_lstm_train)
    ds_val_lstm   = TensorDataset(X_lstm_val,   y_lstm_val)
    ds_test_lstm  = TensorDataset(X_lstm_test,  y_lstm_test)

    best_vl = float('inf'); pc = 0; patience = 5
    start   = time.perf_counter()

    for epoch in range(EPOCHS):
        lstm_model.train()
        for Xb, yb in DataLoader(ds_train_lstm, BATCH_SIZE, shuffle=True):
            lstm_optimizer.zero_grad()
            loss = criterion(lstm_model(Xb), yb.unsqueeze(1))
            loss.backward(); lstm_optimizer.step()

        lstm_model.eval()
        vl_list, va_list = [], []
        with torch.no_grad():
            for Xb, yb in DataLoader(ds_val_lstm, BATCH_SIZE):
                out = lstm_model(Xb)
                vl_list.append(criterion(out, yb.unsqueeze(1)).item())
                va_list.append(((torch.sigmoid(out) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())
        avg_vl = np.mean(vl_list)
        print(f'  Epoch {epoch+1} | Val Loss: {avg_vl:.4f} | Val Acc: {np.mean(va_list):.4f}')
        if avg_vl < best_vl:
            best_vl = avg_vl; pc = 0
            torch.save(lstm_model.state_dict(), os.path.join(OUTPUT_DIR, 'lstm_best.pth'))
        else:
            pc += 1
            if pc >= patience: print('  Early stop.'); break

    lstm_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'lstm_best.pth')))
    lstm_model.eval()
    ta_list = []
    with torch.no_grad():
        for Xb, yb in DataLoader(ds_test_lstm, BATCH_SIZE):
            ta_list.append(((torch.sigmoid(lstm_model(Xb)) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())

    runtime = time.perf_counter() - start
    all_results['LSTM'] = {'val_accuracy': np.mean(va_list), 'test_accuracy': np.mean(ta_list), 'runtime_seconds': runtime}
    print(f'  Test acc: {np.mean(ta_list):.4f} | Time: {runtime:.1f}s')

## Classifier 6: Transformer (Attention Head)

Self-attention over frame-level features [T, 768], then mean-pool and classify.

In [ ]:
if RUN_ATTN:
    if not RUN_LSTM:
        print('LSTM features not available. Skipping Transformer (needs same frame-level features).')
        RUN_ATTN = False

In [ ]:
if RUN_ATTN:
    class TransformerClassifier(nn.Module):
        def __init__(self, d_model=768, nhead=8, num_layers=2, dropout=0.1):
            super().__init__()
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead,
                dim_feedforward=1024, dropout=dropout,
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
            self.classifier  = nn.Linear(d_model, 1)

        def forward(self, x):  # x: [batch, T, 768]
            out = self.transformer(x)    # [batch, T, 768]
            out = out.mean(dim=1)        # [batch, 768] -- mean pool over time
            return self.classifier(out)  # [batch, 1]

    attn_model     = TransformerClassifier()
    attn_optimizer = optim.Adam(attn_model.parameters(), lr=LEARNING_RATE)
    criterion      = nn.BCEWithLogitsLoss()

    best_vl = float('inf'); pc = 0; patience = 5
    start   = time.perf_counter()

    for epoch in range(EPOCHS):
        attn_model.train()
        for Xb, yb in DataLoader(ds_train_lstm, BATCH_SIZE, shuffle=True):
            attn_optimizer.zero_grad()
            loss = criterion(attn_model(Xb), yb.unsqueeze(1))
            loss.backward(); attn_optimizer.step()

        attn_model.eval()
        vl_list, va_list = [], []
        with torch.no_grad():
            for Xb, yb in DataLoader(ds_val_lstm, BATCH_SIZE):
                out = attn_model(Xb)
                vl_list.append(criterion(out, yb.unsqueeze(1)).item())
                va_list.append(((torch.sigmoid(out) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())
        avg_vl = np.mean(vl_list)
        print(f'  Epoch {epoch+1} | Val Loss: {avg_vl:.4f} | Val Acc: {np.mean(va_list):.4f}')
        if avg_vl < best_vl:
            best_vl = avg_vl; pc = 0
            torch.save(attn_model.state_dict(), os.path.join(OUTPUT_DIR, 'attn_best.pth'))
        else:
            pc += 1
            if pc >= patience: print('  Early stop.'); break

    attn_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'attn_best.pth')))
    attn_model.eval()
    ta_list = []
    with torch.no_grad():
        for Xb, yb in DataLoader(ds_test_lstm, BATCH_SIZE):
            ta_list.append(((torch.sigmoid(attn_model(Xb)) > 0.5).float() == yb.unsqueeze(1)).float().mean().item())

    runtime = time.perf_counter() - start
    all_results['Transformer'] = {'val_accuracy': np.mean(va_list), 'test_accuracy': np.mean(ta_list), 'runtime_seconds': runtime}
    print(f'  Test acc: {np.mean(ta_list):.4f} | Time: {runtime:.1f}s')

## Summary Table

In [ ]:
print('=== EXP 2 SUMMARY ===')
print(f'{"Classifier":<20} {"Val Acc":>10} {"Test Acc":>10} {"Runtime (s)":>12}')
print('-' * 55)
for name, res in all_results.items():
    print(f'{name:<20} {res["val_accuracy"]:>10.4f} {res["test_accuracy"]:>10.4f} {res["runtime_seconds"]:>12.1f}')

# Save all results
results_path = os.path.join(OUTPUT_DIR, 'all_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=4)
print(f'\nSaved to {results_path}')